# AMD-Latent
### Encoding images using Visual Encoders

---

The current notebook extracts visual representations from frozen models, saving them and their lower-dimension projection selected.

# Packages import and functions defining

Loading the required packages and definition of functions.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Jupyter packages
import ipywidgets as widgets
from amdlatent.utils.widgets import toggleoptions, filechooser

# Dataset packages
from amdlatent.dataset.audit import naneval
from amdlatent.dataset.extraction import getimages, filteringimages

# RET CLIP packages
from amdlatent.models.encoders import modeldef, extrvencoder, extrlatents, extrvencoderclip, extrvencoderresnet

# Statistical analysis packages
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.preprocessing import MinMaxScaler

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

dirselect = filechooser("Select the <b>AMD Study</b> folder:")
clipselect = toggleoptions(['RETCLIP', 'CLIP', 'ResNet50'], "Select the <b>model</b> to use:")
infoselect = toggleoptions(['None', 'NaN', 'View'], "Select the <b>dataset information</b> to use for filtering:")

eye_box = widgets.Output()
display(eye_box)
eyeselect = None
def _updateeyetoggle(change=None):
    global eyeselect
    eye_box.clear_output()
    if infoselect.value == "Eye":
        with eye_box:
            eyeselect = toggleoptions(
                ['LE', 'RE'],
                "Select the <b>eye type</b> to exclude:"
            )
    else: eyeselect = None
infoselect.observe(_updateeyetoggle, names="value")
_updateeyetoggle()

view_box = widgets.Output()
display(view_box)
viewselect = None
def _updateviewtoggle(change=None):
    global viewselect
    view_box.clear_output()
    if infoselect.value == "View":
        with view_box:
            viewselect = toggleoptions(
                ['F2_LS', 'F2_RS'],
                "Select the <b>view type</b> to exclude:"
            )
    else: viewselect = None
infoselect.observe(_updateviewtoggle, names="value")
_updateviewtoggle()

visualselect = toggleoptions(['PCA', 'tSNE', 'UMAP'], "Select the <b>dimensionality reduction</b> approach:")

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "directory": dirselect.selected_path,
    "cliptype": clipselect.value,
    "option": infoselect.value,
    "blacklist": eyeselect.value if eyeselect is not None else viewselect.value if viewselect is not None else None,
    "visuals": visualselect.value,
}

dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')

if metadata["cliptype"] == "CLIP":
    vencoder = extrvencoderclip()
elif metadata["cliptype"] == "RETCLIP":
    model, ckpt = modeldef()
    vencoder = extrvencoder(model, ckpt)
elif metadata["cliptype"] == "ResNet50":
    vencoder = extrvencoderresnet()

selectedimages = getimages(os.path.join(metadata["directory"], "Data", "EDO"))
latents, paths = extrlatents(selectedimages, vencoder, modeltype=metadata["cliptype"])

if not os.path.exists(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower())):
    os.makedirs(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower()))
np.save(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "latentvec.npy"), latents)
np.save(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "imagepaths.npy"), paths)
paths = pd.DataFrame(paths)

if metadata["option"] == 'NaN':
    blacklist = naneval(os.path.join(metadata["directory"], "Data", "EDO"), dataset)
elif metadata["option"] == 'View':
    blacklist = metadata["blacklist"]
else: blacklist = None

filtpaths = filteringimages(paths, blacklist)
latents = latents[[i for i, path in enumerate(paths[paths.columns[0]].tolist()) if path in filtpaths[filtpaths.columns[0]].tolist()]]

scaler = MinMaxScaler()
scaler.fit(latents)
latents = scaler.transform(latents)

if metadata["visuals"] == 'PCA':
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(latents)
elif metadata["visuals"] == 'tSNE':
    tsne = TSNE(n_components=2)
    reduced = tsne.fit_transform(latents)
elif metadata["visuals"] == 'UMAP':
    mapper = umap.UMAP().fit(latents)
    reduced = mapper.embedding_

np.save(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), f"{metadata['visuals']}_{metadata['option']}.npy"), reduced)